In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 — Unity Catalog Governance
# MAGIC **Layer:** Governance | PII tagging, row-level security, audit logging
# MAGIC
# MAGIC - Column-level PII tags registered in `system.information_schema.column_tags`
# MAGIC - Dynamic view enforces group-based card masking at query time
# MAGIC - Append-only audit log for pipeline observability

# COMMAND ----------

CATALOG      = "fintech_lakehouse_dev"
SCHEMA       = "transactions"
GOLD_TABLE   = f"{CATALOG}.{SCHEMA}.gold_transactions"
SILVER_CORE  = f"{CATALOG}.{SCHEMA}.silver_core_banking"
AUDIT_LOG    = f"{CATALOG}.{SCHEMA}.pipeline_audit_log"
ANALYST_VIEW = f"{CATALOG}.{SCHEMA}.gold_transactions_analyst_view"

print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")

In [0]:
# COMMAND ----------
# Apply PII column tags via Unity Catalog ALTER TABLE

spark.sql(f"""
    ALTER TABLE {GOLD_TABLE}
    ALTER COLUMN card_last_four
    SET TAGS ('pii' = 'true', 'sensitivity' = 'confidential', 'data_class' = 'payment_card')
""")

spark.sql(f"""
    ALTER TABLE {GOLD_TABLE}
    ALTER COLUMN merchant_name
    SET TAGS ('pii' = 'false', 'sensitivity' = 'internal', 'data_class' = 'merchant_data')
""")

spark.sql(f"""
    ALTER TABLE {SILVER_CORE}
    ALTER COLUMN card_last_four
    SET TAGS ('pii' = 'true', 'sensitivity' = 'confidential', 'data_class' = 'payment_card')
""")

print("PII tags applied to gold_transactions and silver_core_banking")

In [0]:
# COMMAND ----------
# Verify tags in system catalog
tag_audit = spark.sql(f"""
    SELECT
        catalog_name, schema_name, table_name,
        column_name, tag_name, tag_value
    FROM system.information_schema.column_tags
    WHERE catalog_name = '{CATALOG}'
    ORDER BY table_name, column_name
""")

tag_audit.display()

In [0]:
# COMMAND ----------
# Row-level security via dynamic view
# card_last_four masked for users outside fintech_payments_analysts group
# High-value transactions hidden from non-privileged users

spark.sql(f"""
    CREATE OR REPLACE VIEW {ANALYST_VIEW} AS
    SELECT
        transaction_id,
        transaction_type,
        amount,
        currency,
        merchant_name,
        status,
        reconciliation_status,
        transaction_date,
        is_high_value,
        gold_processed_at,
        CASE
            WHEN is_account_group_member('fintech_payments_analysts')
            THEN card_last_four
            ELSE '****'
        END AS card_last_four
    FROM {GOLD_TABLE}
    WHERE
        is_account_group_member('fintech_payments_analysts')
        OR is_high_value = false
""")

print(f"Row-level security view created: {ANALYST_VIEW}")

In [0]:
# COMMAND ----------
# Verify view schema and row count matches gold table
view_count = spark.sql(f"SELECT COUNT(*) FROM {ANALYST_VIEW}").collect()[0][0]
gold_count = spark.sql(f"SELECT COUNT(*) FROM {GOLD_TABLE}").collect()[0][0]

print(f"View row count:       {view_count}")
print(f"Gold table row count: {gold_count}")
print(f"Row counts match:     {view_count == gold_count}")

spark.sql(f"DESCRIBE TABLE {ANALYST_VIEW}").display()

In [0]:
# COMMAND ----------
# Write pipeline audit log entry
dq_summary = spark.sql(f"""
    SELECT
        '{GOLD_TABLE}' AS table_name,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT transaction_date) AS date_partitions,
        ROUND(SUM(amount), 2) AS total_amount,
        SUM(CASE WHEN reconciliation_status = 'MATCHED'      THEN 1 ELSE 0 END) AS matched_count,
        SUM(CASE WHEN reconciliation_status = 'MISSING_AUTH' THEN 1 ELSE 0 END) AS missing_auth_count,
        SUM(CASE WHEN is_high_value THEN 1 ELSE 0 END) AS high_value_count,
        current_timestamp() AS audit_timestamp
    FROM {GOLD_TABLE}
""")

dq_summary.write.format("delta") \
    .mode("append") \
    .saveAsTable(AUDIT_LOG)

print(f"Audit log entry written → {AUDIT_LOG}")
dq_summary.display()

In [0]:
%sql
-- COMMAND ----------
%sql
-- Full Delta history — audit trail for compliance
DESCRIBE HISTORY fintech_lakehouse_dev.transactions.gold_transactions